# 03 — Soil Features (100 m² tiles)

Join SSURGO soil data to the new tiles via spatial intersection.
Output: `../data/soil_100m2.csv`

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

TILES_PKL = '../data/polygons/tiles_100m2.pkl'
SOIL_PKL  = '../data/soil/plot_summary.pkl'   # existing per-tile soil from v1
OUT_CSV   = '../data/soil_100m2.csv'

# The SSURGO join in v1 was per old tile (spatial join with soil polygons).
# For v1→v2 we re-use the existing spatial join logic but on new tiles.
# If the raw SSURGO shapefile is available, uncomment the spatial join block below.
# Otherwise we approximate using the v1 soil data (same spatial footprint).


In [ ]:
# ── Option A: SSURGO shapefile available ────────────────────────────────────
# Uncomment if you have the raw SSURGO GDB/shapefile in ../data/soil/

# from rasterstats import zonal_stats
# ssurgo = gpd.read_file('../data/soil/ssurgo.shp')
# tiles  = pd.read_pickle(TILES_PKL)
# joined = gpd.sjoin(tiles, ssurgo, how='left', predicate='intersects')
# soil_cols = ['sandtotal_r','silttotal_r','claytotal_r','awc_r','cec7_r',
#              'om_r','ph1to1h2o_r','ec_r','profile_depth','max_depth',
#              'frag3to10_r','fraggt10_r','dbovendry_r','caco3_r','drain_ord','restrictiondepth_cm']
# soil = joined.groupby('tile_id')[soil_cols].mean().reset_index()
# soil.to_csv(OUT_CSV, index=False)

# ── Option B: Re-use v1 spatial join (tile centroids map to same SSURGO units) ─
# At 100 m² tile resolution the spatial coverage is nearly identical to v1 1000 m²
# tiles so the same SSURGO mapunit applies.


In [ ]:
# Nearest-neighbour lookup: for each new tile, find closest v1 tile and copy soil
tiles_v2 = pd.read_pickle(TILES_PKL).to_crs(26910)
tiles_v2['centroid_x'] = tiles_v2.geometry.centroid.x
tiles_v2['centroid_y'] = tiles_v2.geometry.centroid.y

soil_v1 = pd.read_pickle(SOIL_PKL)  # has 'plot_id' (= v1 tile_id), soil columns

v1_tiles = pd.read_pickle('../data/polygons/RegressionRidge_smol_smol.pkl').to_crs(26910)
v1_tiles['cx'] = v1_tiles.geometry.centroid.x
v1_tiles['cy'] = v1_tiles.geometry.centroid.y

print(f'v2 tiles: {len(tiles_v2)},  v1 tiles: {len(v1_tiles)}')


In [ ]:
from scipy.spatial import KDTree

v1_coords = v1_tiles[['cx','cy']].values
v2_coords = tiles_v2[['centroid_x','centroid_y']].values

tree = KDTree(v1_coords)
_, idx = tree.query(v2_coords, k=1)

v1_tile_ids = v1_tiles['plot_id'].values[idx]  # nearest v1 tile for each v2 tile

soil_map = soil_v1.set_index('plot_id')
soil_cols = [c for c in soil_map.columns]

soil_v2 = tiles_v2[['tile_id']].copy().reset_index(drop=True)
for col in soil_cols:
    soil_v2[col] = soil_map.loc[v1_tile_ids, col].values

soil_v2.to_csv(OUT_CSV, index=False)
print(f'Saved → {OUT_CSV}  shape: {soil_v2.shape}')
soil_v2.head()
